# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and analyzing the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
This dataset follows the [Croissant](https://mlcommons.org/croissant/) schema and is accessible via:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {getattr(dataset.metadata, 'name', 'N/A')}")
print(f"Description:  {getattr(dataset.metadata, 'description', 'N/A')}")

## 2. Data Overview
Let's inspect the available record sets, their `@id`s, and their fields as defined by the Croissant schema. We will enumerate all record sets, fields, and columns and print out their `@id` identifiers for reference in data extraction and analysis.

In [ ]:
# List RecordSets and associated Fields/Columns by @id
def print_record_sets_and_fields(ds):
    record_sets = getattr(ds.metadata, 'record_sets', None)
    if not record_sets:
        print('No record sets found in metadata. Attempting to scan top-level attributes...')
        try:
            # For recent schemas, use dataset.record_sets()
            record_sets = list(ds.record_sets())
        except Exception as e:
            print('No record sets found.')
            return
    if not record_sets:
        print('No record sets found.')
        return
    print("Available Record Sets (by @id):")
    for idx, rs in enumerate(record_sets):
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', f'RS_{idx}')
        rs_name = getattr(rs, 'name', 'N/A')
        print(f"  - @id: {rs_id} (name: {rs_name})")
        fields = getattr(rs, 'fields', getattr(rs, 'field', []))
        if fields:
            print("    Fields/Columns:")
            for f in fields:
                f_id = getattr(f, '@id', None) or getattr(f, 'id', 'N/A')
                f_name = getattr(f, 'name', 'N/A')
                print(f"      - @id: {f_id} (name: {f_name})")
        else:
            print("    (No fields/columns listed in this RecordSet)")
            # For some schemas, fields are not attached
    print("\n")

print_record_sets_and_fields(dataset)

## 3. Data Extraction
We will load the data from one or more record sets using their `@id` as identified above. Each DataFrame's columns will correspond to the record fields' `@id`s.

Note: If there are multiple record sets, you can update the `record_sets` list to include whichever are relevant for your analysis.

In [ ]:
# Identify available record set @ids (edit this after running the above cell if needed):

# Example: record_sets = ["cr:ExperimentDataset"] or the printed @id from above
# We'll use the first discovered record set for demonstration.
def find_first_record_set(ds):
    try:
        record_sets = getattr(ds.metadata, 'record_sets', None)
        if record_sets and len(record_sets) > 0:
            rs = record_sets[0]
            rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
            return rs_id
        # For some schemas, can iterate ds.record_sets()
        rs_list = list(ds.record_sets())
        if len(rs_list) > 0:
            rs = rs_list[0]
            rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
            return rs_id
    except Exception:
        pass
    return None

record_set_id = find_first_record_set(dataset)
if record_set_id is None:
    raise ValueError('No RecordSet @id found in metadata. Please set record_set_id manually.')

record_sets = [record_set_id]
dataframes = {}
for record_set in record_sets:
    print(f"Loading records from RecordSet: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if len(records) == 0:
        print(f"No records loaded for RecordSet {record_set}.")
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps. We'll select a numeric field (by `@id`) from your chosen record set. See the column listing above for possible options.

* Filter records based on a numeric field threshold
* Normalize the numeric field
* Group by a category field

**Edit the field names below with appropriate @ids from your schema if needed.**

In [ ]:
# Update as appropriate after inspecting columns above:
df = dataframes[record_set_id]
# Example guess for numeric field (update as required)
candidate_numeric_fields = [col for col in df.columns if df[col].dtype.kind in "if"]
if not candidate_numeric_fields:
    # Try to coerce columns to numeric to look for numeric
    possible_numeric = []
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().sum() > 0:
                possible_numeric.append(col)
        except Exception:
            continue
    candidate_numeric_fields = possible_numeric

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]  # Choose the first one, or update this
else:
    print('No numeric field detected; please update numeric_field manually.')
    numeric_field = None
# Example: group_field = 'cr:SomeCategoricalFieldId'
possible_group_fields = [col for col in df.columns if col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None

if numeric_field:
    # Demonstrate filtering, normalization, grouping
    # Remove missing for the analysis
    filtered_df = df.copy()
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    filtered_df = filtered_df.dropna(subset=[numeric_field])
    threshold = filtered_df[numeric_field].mean()
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship to the (optional) group field. We'll use matplotlib and seaborn for basic plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # Boxplot by group field if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load Croissant-based metadata and records using `mlcroissant`
- Inspect available record sets and field `@id`s
- Extract data from a chosen record set
- Perform basic exploratory data analysis, normalization, and group-wise aggregation
- Visualize data distributions and relationships

**To extend this analysis:** explore additional record sets or fields, join multiple data sources, and apply domain-specific analytics as needed.